In [1]:
import os 
# Get the current working directory
current_dir = os.getcwd()
print("Current working directory:", current_dir)

import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm

Current working directory: /gpfs/commons/home/kisaev/Leaflet-analysis/tabula_muris_brain/data_processing


In [2]:
import sys
sys.path.append('../../utils')
from functions import * 

### Get featureCounts output and metadata

In [ ]:
# Get input directory where all featureCounts outputs are in:
input_dir = '/gpfs/commons/groups/knowles_lab/data/tabula_muris/smart_seq/featureCounts'
print("The input directory with featureCounts output is: " + input_dir)
print_separator()

# Set specific organ of interest:
organ = 'Muscle'
print("The organ of interest is: " + organ)
print_separator()

# Collect all the *_counts.txt files in the input directory that have the organ of interest in the file name:
files = [os.path.join(input_dir, f) for f in os.listdir(input_dir) if organ in f and f.endswith('_counts.txt')]
print("The number of files in the input directory with tissue of interest is: " + str(len(files)))
print_separator()

# Metadata file:
metadata_file = '/gpfs/commons/groups/knowles_lab/data/tabula_muris/smart_seq/d8a0c006-2991-5ddb-8ed8-7b7e8c187fe3/all_tabula_muris_samples_w_celltype_column.txt'
metadata = pd.read_csv(metadata_file, sep='\t', index_col=0)
print("Loaded metadata file!")
print_separator()

# Subset the metadata file to only include the organ of interest:
metadata = metadata[metadata['tissue'].str.contains(organ)]
print("The number of cells in the metadata corresponding to the organ of interest is: " + str(len(metadata.index)))
print_separator()

cells_freq = metadata.groupby(['tissue', 'subtissue', 'cell_ontology_class']).size()
print("The breakdown of cell types in the organ of interest is: ")
print(cells_freq)
print_separator()

print("The mice used in the organ of interest are: " + str(metadata['mouse.id'].unique()))

In [ ]:
# read each featurecounts output file, add new column to indicate the cell type which can be found after '/gpfs/commons/groups/knowles_lab/data/tabula_muris/smart_seq/featureCounts/' and before _cell_counts.txt'
# save the new matrix to a file
counts_list = []

for file in tqdm(files):
    df = get_featureCounts(file)
    counts_list.append(df)

print("Done reading gene counts for each Brain tissue/cell type")

In [ ]:
# to get PCA need gene expression matrix (rows = genes, columns = cells)

counts_matrix = pd.concat(counts_list, axis=1)
num_cells_found = counts_matrix.shape[1]
num_cells_metadata = len(metadata.index)

print("The number of cells found in the featureCounts files is: " + str(num_cells_found))
print_separator()
print("The number of cells in the metadata file is: " + str(num_cells_metadata))
print_separator()
print("The number of cells in the metadata file that were not found in the featureCounts files is: " + str(num_cells_metadata - num_cells_found))
print_separator()

### Make anndata object

In [ ]:
# Generate adata object from featureCounts output and metadata file
X = counts_matrix.values.T
adata = ad.AnnData(X, dtype=X.dtype)

# add cell IDs and gene names to adata object
adata.obs_names = counts_matrix.columns
adata.var['gene_name'] = counts_matrix.index
adata.var_names = counts_matrix.index

# ensure cells in metadata are in the same order as in adata
metadata = metadata.loc[adata.obs_names]
metadata["cell_id"] = metadata.index
# remove index from metadata
metadata = metadata.reset_index(drop=True)
# remove free_annotation column from metadata since it doesn't contain any info 
metadata = metadata.drop(columns=['free_annotation'])
# add metadata 
adata.obs = metadata

In [ ]:
# save adata object for further downstream analysis
adata_path = '/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/'
# add organ to file name for output
adata.write_h5ad(os.path.join(adata_path, 'adata_' + organ + '.h5ad'))
print("Saved adata object to: " + adata_path + "adata_" + organ + ".h5ad") 